In [0]:
from pyspark.sql.functions import row_number,col,countDistinct,min,max
from pyspark.sql.window import Window

data=[('Aarav','2024-01-01'),('Aarav','2024-01-02'),('Aarav','2024-01-03'),('Aarav','2024-01-06'),('Aarav','2024-01-07'),('Priya','2024-01-10'),('Priya','2024-01-02'),('Priya','2024-01-03'),('Priya','2024-01-04'),('Priya','2024-01-05')]
schema=(['user_id','login_date'])
df = spark.createDataFrame(data,schema)
window = Window.partitionBy("user_id").orderBy("login_date")
df_cons_dates = df.withColumn("grp", col("login_date").cast("date") - row_number().over(window))

df_streak = df_cons_dates.groupBy("user_id","grp").agg(
    countDistinct("login_date").alias("streak"),
    min("login_date").alias("start_date"),
    max("login_date").alias("end_date")    
).filter(col("streak") >= 3)

df_strk_days = df_streak.select("user_id","start_date","end_date","streak")
df_strk_days.show()

df_strk_days.explain(True)

In [0]:
from pyspark.sql.functions import row_number,col,count,min,max
from pyspark.sql.window import Window

data=[('Aarav','2024-01-01'),('Aarav','2024-01-02'),('Aarav','2024-01-03'),('Aarav','2024-01-06'),('Aarav','2024-01-07'),('Priya','2024-01-10'),('Priya','2024-01-02'),('Priya','2024-01-03'),('Priya','2024-01-04'),('Priya','2024-01-05')]
schema=(['user_id','login_date'])
df = spark.createDataFrame(data,schema).repartition("user_id")

window = Window.partitionBy("user_id").orderBy("login_date")
df_cons_dates = df.withColumn("grp", col("login_date").cast("date") - row_number().over(window))

df_streak = df_cons_dates.groupBy("user_id","grp").agg(
    count("*").alias("streak"),
    min("login_date").alias("start_date"),
    max("login_date").alias("end_date")    
).filter(col("streak") >= 3)

df_strk_days = df_streak.select("user_id","start_date","end_date","streak")
df_strk_days.show()

df_strk_days.explain(True)

In [0]:
from pyspark.sql.functions import row_number,col,count,min,max
from pyspark.sql.window import Window

data=[(1, 'India', 'Australia', 'India'),(2,'England','Australia','England'),
	(3, 'India', 'Australia', 'India'),(4,'England','Australia','England'),
	(5,'SouthAfrica','India','SouthAfrica'),(6,'NewZealand','Australia','NewZealand'),(7, 'India', 'Australia', 'India'),(8,'England','NewZealand','NewZealand'),
	(9, 'NewZealand', 'Australia', 'NewZealand'),(10,'England','NewZealand','England'),
	(11,'SouthAfrica','NewZealand','SouthAfrica'),(12,'England','Australia','NewZealand')]
schema=(['id', 'team_1', 'team_2', 'winner'])
df = spark.createDataFrame(data,schema)

df_union = (df.select(col("team_1").alias('team'),col('winner')).
unionAll(df.select(col('team_2').alias('team'),col('winner')))
)

from pyspark.sql.functions import *

df_board = df_union.groupBy("team").agg(

    count("*").alias("played"),

    sum(
        when(col("team") == col("winner"), 1).otherwise(0)
    ).alias("won"),

    sum(
        when(col("team") != col("winner"), 1).otherwise(0)
    ).alias("lost")

).withColumn(
    "points",
    col("won") * 2
)

df_board.show()

In [0]:
from pyspark.sql.functions import row_number,col,count,min,max
from pyspark.sql.window import Window

data=[(101, 'Aarav', '2024-01-05', 500),(102,'Aarav','2024-01-12',700),
	(103, 'Aarav', '2024-03-20',100),(104,'Aarav','2024-03-25',500),
	(105,'Priya','2024-02-01',800),(106,'Priya','2024-02-15',500),
	(107, 'Priya', '2024-02-28',700),(108,'Priya','2024-03-10',500),
	(109, 'Enzo', '2024-01-05',500)]
schema=(['order_id', 'cust_id', 'order_date', 'amount'])
df = spark.createDataFrame(data,schema)

window = Window.partitionBy("cust_id").orderBy("order_date")

df_gap = df.select(col("cust_id"),
                   col("order_date").alias("gap_end"),
                    lag("order_date",1).over(window).alias("gap_start"),
                    datediff(col("order_date").cast('date'), lag("order_date",1).over(window).cast('date')).alias("gap_days")
                    )

window_rank = Window.partitionBy("cust_id").orderBy(col("gap_days").desc())

df_ranked = df_gap.filter(col("gap_days").isNotNull()).withColumn("rn",
                    row_number().over(window_rank)
                    )

df_longest_gap = df_ranked.filter(
    col("rn") == 1
).select(
    "cust_id",
    col("gap_days").alias("long_gap_days"),
    "gap_start",
    "gap_end"
)

df_longest_gap.show()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

data=[(1,'War','great 3D','8.9'),(2,'science','fiction','9.9'),(3,'Irish','boring','7.9'),(4,'Ice song','Fantacy','2.9'),(5,'House card','Interesting','3.9')]
schema=(['id','movie','description','rating'])
df = spark.createDataFrame(data,schema)

df_boring = df.filter(col('id')%2!=0).filter(~col('description').contains('boring')).orderBy(col('rating').desc())

df_result = df_boring.select('*')
df_result.show()

In [0]:
from pyspark.sql.functions import col,count
from pyspark.sql.window import Window

data=[(1,1),(2,2),(3,3),(4,3)]
schema=(['order_number','customer_number'])
df = spark.createDataFrame(data,schema)

df_agg = df.groupBy(col('customer_number')).agg(count('*').alias("count")).orderBy(col('count').desc()).limit(1)

df_result = df_agg.select(col("customer_number").alias("id"))

df_result.show()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

data=[(101,1,2000,'2024-01-12'),(102,1,2400,'2024-01-15'),(103,2,1000,'2024-01-15'),(104,2,1200,'2024-01-16'),(105,3,4000,'2024-01-15'),(106,3,1400,'2024-02-15'),(107,4,1400,'2024-01-25'),(108,4,1270,'2024-01-26')]
schema=(['order_id','cid','amount','order_date'])
df = spark.createDataFrame(data,schema)

data1 = [(1,'Aarav','Mumbai'),(2,'Priya','Delhi'),(3,'Enzo','Chennai'),(4,'Bruno','Kerala')]
schema1 = ['id','name','city']
df1 = spark.createDataFrame(data1,schema1)

df_join = df.join(df1,df.cid==df1.id,'inner').groupBy('id','name','city').agg(sum(col('amount')).alias('total_spent'))

df_select = df_join(select('name'))

df_select.show()